# Notebook 06 — Exceptions & context managers

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate → advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `05-classes-oop.ipynb`

**Next up:** `07-decorators-generators.ipynb`

---

## Learning objectives

- Raise and catch exceptions narrowly.
- Preserve tracebacks with `raise ... from`.
- Author reusable setup/teardown with context managers.

## Table of contents

1. **Try / except patterns**
2. **Chaining exceptions**
3. **`contextmanager` timer**
4. **Progressive drills — coercion → chained causes → class CM**
5. **Exercise — `nullsafe_get`**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Narrow catches

*Explanation:* Catch `json.JSONDecodeError`, not blanket `Exception`, unless at process boundary.


In [ ]:
import json

def loads_obj(raw: str) -> dict:
    try:
        val = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError("bad json") from e
    if not isinstance(val, dict):
        raise TypeError("expected object")
    return val


## 2 · Context manager timer

*Explanation:* Mirrors dependency scopes in FastAPI — acquire, yield, release.


In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def span(name: str):
    t0 = time.perf_counter()
    yield
    print(f"[{name}] {1000*(time.perf_counter()-t0):.1f} ms")

with span("retrieval"):
    time.sleep(0.02)


---

## Progressive drills — **easy → harder**

Failures are control flow in AI stacks—practice **narrow catches**, **cause chains**, and **scoped teardown**.


### A · Easiest — coerce tool ints safely

LLMs emit `"24"` strings—convert defensively.


In [ ]:
def coerce_positive_int(raw: str) -> int:
    try:
        value = int(raw)
    except ValueError as exc:
        raise ValueError(f"not an int: {raw!r}") from exc
    if value <= 0:
        raise ValueError("must be positive")
    return value


print(coerce_positive_int("128"))
try:
    coerce_positive_int("twelve")
except ValueError as err:
    print("blocked:", err)


### B · Medium — translate KeyErrors into domain errors

Nested dict navigation mirrors messy JSON tool payloads.


In [ ]:
def extract_tool_name(payload: dict) -> str:
    try:
        return payload["tool"]["name"]
    except KeyError as exc:
        raise ValueError("payload missing tool.name path") from exc


sample = {"tool": {"name": "calculator", "args": {}}}
print(extract_tool_name(sample))
try:
    extract_tool_name({"tool": {}})
except ValueError as err:
    print("blocked:", err.__cause__)


### C · Harder — class-based context manager

Beyond `@contextmanager`, explicit `__enter__`/`__exit__` maps to SDK sessions.


In [ ]:
class TraceSpan:
    def __init__(self, label: str) -> None:
        self.label = label

    def __enter__(self) -> "TraceSpan":
        print("OPEN", self.label)
        return self

    def __exit__(self, exc_type, exc, tb) -> bool:
        print("CLOSE", self.label)
        return False  # propagate exceptions


with TraceSpan("embed-batch"):
    print("doing work...")


### Exercise — `nullsafe_get`

Function `nullsafe_get(d, key)` returns `d[key]` if `key` exists **and** value is not `None`; otherwise returns `""`. Raises **nothing**.


In [ ]:
def nullsafe_get(d: dict, key: str) -> str:
    raise NotImplementedError


assert nullsafe_get({"a": None}, "a") == ""
assert nullsafe_get({"a": "x"}, "a") == "x"
assert nullsafe_get({}, "missing") == ""
print("OK")


### Solution — nullsafe_get

<details>
<summary>Click to expand</summary>

```python
def nullsafe_get(d: dict, key: str) -> str:
    if key not in d:
        return ""
    val = d[key]
    return "" if val is None else val
```

</details>
